# Preprocessing — Steam Top 1000 (2024–2026)

Il notebook legge i dati grezzi da `data/`, li ispeziona, documenta i problemi
trovati, applica le correzioni e salva un file pulito in `preprocessed/`.

Il file `preprocessing.ipynb` porta i dati nel formato desiderato.

L'analisi si trova in `analysis.ipynb`, che legge solo il file pulito: i dati grezzi
non vengono mai modificati.

**Fonte:** *Top 1000 Steam Games (2024–2026)*, Kaggle — raccolto a marzo 2026
dalla Steam Storefront API e da SteamSpy.

Il progetto risponde a tre domande:

**[D1]** Fra i giochi a pagamento, quelli che costano di più hanno recensioni migliori?

**[D2]** Il punteggio delle recensioni varia fra i generi?

**[D3]** I free-to-play reggono ancora il confronto con i giochi a pagamento?

Le scelte di pulizia documentate nel notebook sono motivate rispetto a queste domande.

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)

## 1. Lettura dei dati grezzi

Il file è salvato con BOM (Byte Order Mark) iniziale: senza `encoding="utf-8-sig"`
la prima colonna verrebbe letta come `\ufeffAppID` invece che `AppID` e ogni
accesso a `df["AppID"]` fallirebbe con un `KeyError`.

In [2]:
df = pd.read_csv("data/steam_games_2026.csv", encoding="utf-8-sig")
print(df.shape)
df.head(3)

(1000, 12)


,AppID,Name,Release_Date,Primary_Genre,All_Tags,Price_USD,Discount_Pct,Review_Score_Pct,Total_Reviews,Steam_Deck_Status,Estimated_Owners,24h_Peak_Players
0,730,Counter-Strike 2,2012-08-21,Action,FPS;Shooter;Multiplayer;Competitive;Action;Tea...,0.00,0,83,4980365,Unknown,149410950,1013936
1,2868840,Slay the Spire 2,2026-03-05,Indie,Strategy;Roguelike;Card Game;Deckbuilding;Co-o...,24.99,0,97,49549,Unknown,1486470,0
2,3321460,Crimson Desert,2026-03-19,Action,Action;Open World;Singleplayer;Adventure;Comba...,69.99,0,0,0,Unknown,0,0


In [3]:
# quadro d'insieme: tipo, valori mancanti, valori distinti per colonna
pd.DataFrame({
    "tipo_dati": df.dtypes,
    "dati_mancanti": df.isnull().sum(),
    "dati_distinti": df.nunique(),
})

,tipo_dati,dati_mancanti,dati_distinti
AppID,int64,0,1000
Name,object,0,1000
Release_Date,object,1,744
Primary_Genre,object,0,12
All_Tags,object,14,958
Price_USD,float64,0,99
Discount_Pct,int64,0,32
Review_Score_Pct,int64,0,71
Total_Reviews,int64,0,892
Steam_Deck_Status,object,0,1


## 2. Problemi trovati

L'ispezione ha fatto emergere otto problemi. Vengono documentati uno per uno perché
due di essi cambiano quali domande i dati possono davvero reggere.

### 2.1 `Steam_Deck_Status` non contiene informazione

La colonna ha un solo valore distinto su mille righe.

In [4]:
df["Steam_Deck_Status"].value_counts(dropna=False)

Steam_Deck_Status
Unknown    1000
Name: count, dtype: int64

Il valore è `Unknown` per tutte e mille le righe: è una colonna di **dati mancanti** registrati come stringa.
`isnull()` non ne segnalava nessuno perché per pandas `"Unknown"` è un valore come un altro.

La colonna viene eliminata (sezione 3.1).

### 2.2 `Estimated_Owners` è derivata da `Total_Reviews`

Nella tabella di ricognizione le due colonne hanno lo stesso numero di valori distinti, 892. Il sospetto è che una delle due sia derivata dall'altra e non un dato raccolto indipendentemente.

In [5]:
df[["Total_Reviews", "Estimated_Owners"]].nunique()

Total_Reviews       892
Estimated_Owners    892
dtype: int64

Gli 892 valori distinti in entrambe suggeriscono una corrispondenza biunivoca tra le due colonne, cioè una trasformazione che conserva i valori distinti. La più semplice è la moltiplicazione per una costante: verifico calcolando il rapporto.

In [6]:
dati_validi = df[df["Total_Reviews"] > 0]
rapporto = dati_validi["Estimated_Owners"] / dati_validi["Total_Reviews"]

print("righe valide:", len(dati_validi))
print("valori distinti del rapporto:", rapporto.nunique())
print("rapporto:", rapporto.min(), "→", rapporto.max())

righe valide: 898
valori distinti del rapporto: 1
rapporto: 30.0 → 30.0


Il rapporto vale esattamente 30,0 per tutte le 898 righe valide.

`Estimated_Owners` è quindi sempre uguale a `Total_Reviews * 30` e non aggiunge nessuna nuova informazione.

Per questo motivo la colonna viene eliminata e quando serve una misura della diffusione si usa `Total_Reviews`, specificando che si tratta di un proxy e non del numero di vendite.

**Attenzione:** i valori distinti sono solo un indizio, non una prova. Due colonne diverse potrebbero avere lo stesso numero per caso. La dimostrazione è il rapporto costante.

### 2.3 102 giochi senza dati di recensione

`Total_Reviews == 0` per 102 giochi. Sono valori mancanti registrati come zero: alcuni di questi titoli hanno decine di migliaia di giocatori
simultanei, quindi le recensioni esistono ma non sono state raccolte.

In [7]:
dati_assenti = df["Total_Reviews"] == 0
print("giochi senza recensioni:", dati_assenti.sum())

giochi senza recensioni: 102


Controprova: alcuni hanno moltissimi giocatori attivi

In [8]:
df.loc[dati_assenti, ["Name", "Release_Date", "Total_Reviews", "24h_Peak_Players"]] \
  .sort_values("24h_Peak_Players", ascending=False).head(5)

,Name,Release_Date,Total_Reviews,24h_Peak_Players
13,Marvel Rivals,2024-12-05,0,106611
188,IdleOn,2025-11-06,0,11081
140,MapleStory,2012-08-09,0,8092
234,Palia,2024-03-25,0,5140
950,Shop Titans,2020-05-05,0,3279


Questi mancanti non sono distribuiti a caso.

In [9]:
f2p = df["Price_USD"] == 0

confronto = pd.DataFrame({
    "giochi":           [f2p.sum(), (~f2p).sum()],
    "senza recensioni": [(dati_assenti & f2p).sum(), (dati_assenti & ~f2p).sum()],
}, index=["free-to-play", "a pagamento"])
confronto["quota %"] = (confronto["senza recensioni"] / confronto["giochi"] * 100).round(1)
confronto

,giochi,senza recensioni,quota %
free-to-play,108,56,51.9
a pagamento,892,46,5.2


Il dato manca per il **52% dei free-to-play** contro il **5% dei giochi a
pagamento**.

Eliminare queste righe con `dropna()` dimezzerebbe il campione free-to-play e la conclusione sui free-to-play potrebbe risultare sbagliata.

Queste righe restano nel file pulito e non vengono marcate, la condizione che le identifica (`Numero di recensioni == 0`) è già nei dati. Stessa cosa vale per i free-to-play (`Prezzo (USD) == 0`).

### 2.4 Date: valori non interpretabili e giochi non ancora usciti

I dati sono di marzo 2026, ma alcuni titoli hanno data di uscita successiva:
sono preordini presenti in classifica, senza recensioni né giocatori.

In [10]:
date_rilascio = pd.to_datetime(df["Release_Date"], errors="coerce")
print("date non interpretabili:", date_rilascio.isnull().sum())
print("intervallo:", date_rilascio.min().date(), "→", date_rilascio.max().date())
df.loc[date_rilascio.isnull(), ["Name", "Release_Date"]]

date non interpretabili: 2
intervallo: 2006-11-29 → 2026-07-09


,Name,Release_Date
689,Assassin’s Creed® IV Black Flag™,NaN
926,s&box,April 2026


I valori non interpretabili passano da 1 a 2 dopo la conversione: oltre al NaN già presente, la stringa `April 2026` è leggibile ma priva del giorno, quindi `errors="coerce"` la trasforma in `NaT`. È un mancante che `isnull()` sulla colonna grezza non rilevava.

In [11]:
data_approssimativa_raccolta = pd.Timestamp("2026-03-31")
non_uscito = date_rilascio > data_approssimativa_raccolta
print("non ancora usciti:", non_uscito.sum())

df.loc[non_uscito, ["Name", "Release_Date", "Total_Reviews", "24h_Peak_Players"]]

non ancora usciti: 5


,Name,Release_Date,Total_Reviews,24h_Peak_Players
92,Forza Horizon 6,2026-05-18,0,0
242,Echoes of Aincrad,2026-07-09,0,0
329,PRAGMATA,2026-04-16,0,0
493,LEGO® Batman™: Legacy of the Dark Knight,2026-05-29,0,0
787,007 First Light,2026-05-27,0,0


Le due righe con `NaT` non sono confrontabili: qualunque confronto con `NaT` restituisce `False`, quindi verranno naturalmente escluse da un filtro sulla data senza bisogno di segnalarle esplicitamente.
I preordini si identificano con `Data di uscita > 2026-03-31`.

### 2.5 Valori Unknown in `Primary_Genre`

`All_Tags` è una stringa separata da `;`. 927 giochi su 1000 hanno esattamente
dieci tag.

In [12]:
tags = df["All_Tags"].fillna("").str.split(";")
n_tag = tags.apply(lambda lista: len([t for t in lista if t]))
n_tag.value_counts().sort_index()

All_Tags
0      14
1       7
2       5
3      26
4       7
5       3
6       4
7       4
8       1
9       2
10    927
Name: count, dtype: int64

I dati non permettono di stabilire se il limite sia del Dataset o di Steam.

Per **[D2]** si utilizza la colonna `Primary_Genre` che contiene un solo genere per gioco.

In [13]:
df["Primary_Genre"].value_counts(dropna=False)

Primary_Genre
Action                   579
Adventure                102
Indie                     73
RPG                       71
Casual                    63
Simulation                47
Strategy                  27
Racing                    17
Massively Multiplayer     14
Unknown                    3
Sports                     2
Early Access               2
Name: count, dtype: int64

Sono presenti 3 valori `Unknown` che restano nel file pulito: la condizione che le identifica (`Genere Primario == "Unknown"`) è già nei dati.

### 2.6 Punteggio delle recensioni incoerente con il numero di recensioni

31 giochi hanno `Review_Score_Pct == 0`. Ventisette di questi sono fra i 102
senza recensioni della sezione 2.3, i restanti quattro hanno invece migliaia di recensioni.

In [14]:
punteggio_nullo = df["Review_Score_Pct"] == 0

print("giochi con punteggio 0:", punteggio_nullo.sum())
print("di cui già senza recensioni (sezione 2.3):", (punteggio_nullo & dati_assenti).sum())
print("con recensioni ma punteggio 0:", (punteggio_nullo & ~dati_assenti).sum())

df.loc[punteggio_nullo & ~dati_assenti,
       ["Name", "Primary_Genre", "Review_Score_Pct", "Total_Reviews", "24h_Peak_Players"]]

giochi con punteggio 0: 31
di cui già senza recensioni (sezione 2.3): 27
con recensioni ma punteggio 0: 4


,Name,Primary_Genre,Review_Score_Pct,Total_Reviews,24h_Peak_Players
54,FINAL FANTASY XIV Online,Massively Multiplayer,0,76795,19824
64,Path of Exile,Action,0,1033,4694
156,Lost Ark,Action,0,65259,17355
989,FINAL FANTASY VIII,RPG,0,7730,99


FINAL FANTASY XIV Online ha 76.795 recensioni e 19.824 giocatori simultanei;
Lost Ark ne ha 65.259 e 17.355. Un punteggio dello 0% non è giustificabile con questi
numeri.

L'incoerenza si presenta anche nella direzione opposta.

In [15]:
df.loc[dati_assenti, "Review_Score_Pct"].value_counts().head()

Review_Score_Pct
0     27
78     5
73     4
67     4
68     4
Name: count, dtype: int64

75 dei 102 giochi senza recensioni hanno comunque un punteggio
plausibile, 78%, 73%, 67%, 68% etc.

Le due colonne si contraddicono quindi in entrambe le direzioni, segno che non
provengono dalla stessa interrogazione e che, su queste righe, almeno una delle due è
inaffidabile. È un motivo in più per escludere in analisi l'intero gruppo
individuato nella sezione 2.3.

Le righe restano nel file pulito: la condizione che le identifica (`Recensioni positive (%) == 0` insieme a `Numero di recensioni > 0`) è già nei dati.

### 2.7 Giocatori simultanei a zero su un terzo dei titoli

`24h_Peak_Players` vale 0 per una parte consistente del dataset. In una classifica
dei titoli più venduti su Steam il valore è già poco plausibile e lo diventa ancora
meno guardando quali giochi coinvolge.

In [16]:
senza_giocatori = df["24h_Peak_Players"] == 0

print("giochi con 0 giocatori simultanei:", senza_giocatori.sum())
print("di cui con più di 10.000 recensioni:",
      (senza_giocatori & (df["Total_Reviews"] > 10000)).sum())

df.loc[senza_giocatori, ["Name", "Release_Date", "Total_Reviews", "24h_Peak_Players"]] \
  .sort_values("Total_Reviews", ascending=False).head(8)

giochi con 0 giocatori simultanei: 304
di cui con più di 10.000 recensioni: 45


,Name,Release_Date,Total_Reviews,24h_Peak_Players
344,Hollow Knight: Silksong,2025-09-04,364174,0
8,ARC Raiders,2025-10-30,264150,0
50,Battlefield™ 6,2025-10-10,262749,0
89,PEAK,2025-06-16,243595,0
126,Dispatch,2025-10-22,157216,0
260,Megabonk,2025-09-18,92901,0
830,Escape From Duckov,2025-10-16,80785,0
4,Resident Evil Requiem,2026-02-26,79667,0


Hollow Knight: Silksong ha 364.174 recensioni ed è uscito a settembre 2025; ARC
Raiders ne ha 264.150, Battlefield 6 262.749. Sono fra i titoli più venduti
dell'ultimo anno, usciti da pochi mesi e riportano zero giocatori simultanei.

I casi non sono isolati: 45 righe con giocatori a zero hanno più di 10.000
recensioni e i titoli coinvolti sono in prevalenza **recenti**, non vecchi o
abbandonati. Il valore è incoerente con il numero di recensioni della stessa riga.

A differenza dei casi precedenti, qui il difetto riguarda **un terzo del campione**:
`Giocatori simultanei (picco 24h)` è utilizzabile solo sui restanti due terzi.

Le righe restano nel file pulito e la condizione che le identifica
(`Giocatori simultanei (picco 24h) == 0`) è già nei dati.

### 2.8 `Price_USD` è il prezzo rilevato, non quello di listino

342 giochi hanno uno sconto attivo al momento della raccolta. Per questi il prezzo registrato non è il prezzo del gioco, ma quello del giorno.

In [17]:
scontati = df["Discount_Pct"] > 0

print("giochi scontati:", scontati.sum())

giochi scontati: 342


Il listino è ricostruibile invertendo lo sconto: 

`listino = prezzo rilevato / (1 − sconto / 100)`

La colonna `Price_USD` viene corretta (sezione 3.1) e rinominata in `Prezzo (USD)` (sezione 3.4) mentre la colonna `Discount_Pct` viene cancellata (sezione 3.2) in quanto non servirà nell'analisi.

## 3. Pulizia

Gli interventi si limitano a eliminare ciò che non contiene informazione, convertire i tipi e rendere i nomi delle colonne utilizzabili in analisi.

In [18]:
pulito = df.copy()

### 3.1 Correzzione colonna `Price_USD`

Viene corretta la colonna `Price_USD`, calcolando i prezzi di listino e arrotondandoli tutti all'intero più vicino.

In [19]:
scontati = pulito["Discount_Pct"] > 0
pulito.loc[scontati, "Price_USD"] = (
    pulito.loc[scontati, "Price_USD"] / (1 - pulito.loc[scontati, "Discount_Pct"] / 100)
)

pulito["Price_USD"] = pulito["Price_USD"].round().astype(int)

### 3.2 Colonne che non contengono informazione

Si eliminano le colonne per cui è già stata motivata l'eliminazione. Inoltre si elimina `Discount_Pct` in quanto non verrà utilizzata per le analisi.

In [20]:
pulito = pulito.drop(columns=["Steam_Deck_Status", "Estimated_Owners", "All_Tags", "Discount_Pct"])

pulito.shape

(1000, 8)

### 3.3 Conversione delle date

La colonna viene convertita da testo a datetime. Il parametro `errors="coerce"` sostituisce con `NaT` i valori non interpretabili (i due individuati nella sezione 2.4). I valori diventano così confrontabili e ordinabili e le due righe problematiche restano identificabili.

In [21]:
pulito["Release_Date"] = pd.to_datetime(pulito["Release_Date"], errors="coerce")

pulito.head(3)

,AppID,Name,Release_Date,Primary_Genre,Price_USD,Review_Score_Pct,Total_Reviews,24h_Peak_Players
0,730,Counter-Strike 2,2012-08-21,Action,0,83,4980365,1013936
1,2868840,Slay the Spire 2,2026-03-05,Indie,25,97,49549,0
2,3321460,Crimson Desert,2026-03-19,Action,70,0,0,0


### 3.4 Nomi delle colonne in italiano, con le unità di misura

`seaborn` usa i nomi delle colonne come etichette degli assi. Rinominandole una
volta sola qui, tutti i grafici escono automaticamente con assi in italiano e
unità di misura, senza dover scrivere `set_xlabel` in ogni figura.

In [22]:
NUOVI_NOMI = {
    "Name":              "Nome",
    "Release_Date":      "Data di uscita",
    "Primary_Genre":     "Genere Primario",
    "Price_USD":         "Prezzo (USD)",
    "Review_Score_Pct":  "Recensioni positive (%)",
    "Total_Reviews":     "Numero di recensioni",
    "24h_Peak_Players":  "Giocatori simultanei (picco 24h)"
}
pulito = pulito.rename(columns=NUOVI_NOMI)

pulito.columns.tolist()

['AppID',
 'Nome',
 'Data di uscita',
 'Genere Primario',
 'Prezzo (USD)',
 'Recensioni positive (%)',
 'Numero di recensioni',
 'Giocatori simultanei (picco 24h)']

### 3.5 Controlli finali

In [23]:
print("righe:", len(pulito))
print("colonne:", len(pulito.columns))
print("AppID duplicati:", pulito["AppID"].duplicated().sum())
print()
print("prezzo negativo:", (pulito["Prezzo (USD)"] < 0).sum())
print("punteggio fuori 0-100:", (~pulito["Recensioni positive (%)"].between(0, 100)).sum())
print()
pulito[["Prezzo (USD)", "Recensioni positive (%)", "Numero di recensioni",
        "Giocatori simultanei (picco 24h)"]].describe().T

righe: 1000
colonne: 8
AppID duplicati: 0

prezzo negativo: 0
punteggio fuori 0-100: 0



,count,mean,std,min,25%,50%,75%,max
Prezzo (USD),1000.0,27.827,19.954881,0.0,14.00,25.0,40.00,100.0
Recensioni positive (%),1000.0,79.658,19.382227,0.0,75.00,85.0,92.00,100.0
Numero di recensioni,1000.0,63871.190,211587.922997,0.0,1638.25,11732.0,48515.75,4980365.0
Giocatori simultanei (picco 24h),1000.0,5213.243,40375.371421,0.0,0.00,286.5,1601.00,1013936.0


## 4. Salvataggio

Il dataset viene salvato in parquet che conserva i tipi senza doverli
reinterpretare a ogni lettura.

In [24]:
OUT = "preprocessed/steam_pulito.parquet"
pulito.to_parquet(OUT, index=False)

verifica = pd.read_parquet(OUT)
print("salvato:", OUT, verifica.shape)
verifica.dtypes

salvato: preprocessed/steam_pulito.parquet (1000, 8)


AppID                                        int64
Nome                                        object
Data di uscita                      datetime64[ns]
Genere Primario                             object
Prezzo (USD)                                 int32
Recensioni positive (%)                      int64
Numero di recensioni                         int64
Giocatori simultanei (picco 24h)             int64
dtype: object

## 5. Riepilogo

| Intervento | Motivo |
|---|---|
| Eliminata `Steam_Deck_Status` | `Unknown` su tutte le righe: nessun valore informativo (sezione 2.1) |
| Eliminata `Estimated_Owners` | pari a `Total_Reviews * 30`: nessuna informazione aggiuntiva (sezione 2.2) |
| Eliminata `All_Tags` | non utilizzata da nessuna delle tre domande, il troncamento ai primi dieci tag la rende comunque inadatta a fare conteggi |
| `Price_USD` ricondotto al prezzo di listino | per 342 giochi il valore registrato era il prezzo scontato del giorno, non quello del gioco (sezione 2.8) |
| Prezzi arrotondati all'intero | assorbe l'errore introdotto dallo sconto espresso in numeri interi, applicato a tutte le righe per uniformità di scala |
| Eliminata `Discount_Pct` | usata per ricostruire il listino (sezione 3.1), poi non più necessaria |
| `Data di uscita` convertita in `datetime` | le date erano stringhe; `errors="coerce"` porta a `NaT` i 2 valori non interpretabili |
| Colonne rinominate in italiano con le unità di misura | i nomi delle colonne diventano le etichette degli assi nei grafici |

Il preprocessing **non aggiunge colonne**: il file pulito contiene 8 colonne e tutte
e **1000** le righe. Nessuna riga è stata eliminata.

### Problemi documentati e non risolti

Le righe problematiche restano nel file pulito. Non vengono marcate perché la
condizione che le identifica è già presente nei dati: un flag ne sarebbe soltanto
una copia sotto un altro nome e una colonna in più da tenere sincronizzata.

| Problema | Righe | Come si identificano |
|---|---|---|
| Giochi senza dati di recensione — **52% dei free-to-play contro 5% dei giochi a pagamento**, mancanza non casuale | 102 | `Numero di recensioni == 0` |
| Date non interpretabili | 2 | `Data di uscita` nulla |
| Preordini non ancora usciti alla data di raccolta | 5 | `Data di uscita > 2026-03-31` |
| Genere principale a Unknown | 3 | `Genere Primario == "Unknown"` |
| Punteggio a 0% con recensioni presenti | 4 | `Recensioni positive (%) == 0` e `Numero di recensioni > 0` |
| Giocatori simultanei a 0 | 304 | `Giocatori simultanei (picco 24h) == 0` |

Chi analizza ricostruisce ciascun gruppo quando gli serve ed è così costretto a
dichiarare cosa sta escludendo.

### Limiti da dichiarare nella presentazione

1. Il campione è la classifica dei mille giochi più venduti a marzo 2026: sono tutti
   titoli di successo. I risultati valgono *fra i giochi che vendono*, non per
   l'intero catalogo Steam.
2. `Giocatori simultanei (picco 24h)` presenta due limiti distinti. È la rappresentazione di un singolo giorno, quello della raccolta dati, quindi rumorosa e favorevole ai titoli che in quella giornata avevano un aggiornamento o un evento. Inoltre è a zero su un terzo del dataset, anche su titoli molto recensiti (sezione 2.7): la colonna è utilizzabile solo sui restanti due terzi.
3. `Numero di recensioni` è un proxy della diffusione, non un dato di vendite.
4. `Numero di recensioni` e `Giocatori simultanei (picco 24h)` lavorano su fasce temporali diverse: la prima accumula dall'uscita del titolo, la seconda è l'immagine del giorno di raccolta dei dati. A parità di picco di giocatori, un gioco uscito dieci anni fa ha accumulato più recensioni di uno uscito l'anno scorso e ottiene quindi un rapporto più basso. L'indicatore risente dell'età del titolo oltre che dell'attività.
5. Il prezzo di listino dei 342 giochi scontati è ricostruito, non rilevato.
   La verifica mostra uno scarto mediano di un centesimo dai prezzi canonici, ma
   resta una stima.